# Zero-DCE++ with Channel Attention
## Low-Light Image Enhancement - Final Project

**Novelty:** SE-Block (Channel Attention) after Conv3
**Targets:** PSNR > 19 dB, SSIM > 0.65, LPIPS < 0.30

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim_metric, peak_signal_noise_ratio as psnr_metric
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA GeForce RTX 3060


In [20]:
# SE-Block (Channel Attention - Your Novelty)
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock, self).__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w

# Zero-DCE++ Model
class DCENet(nn.Module):
    def __init__(self, use_attention=True):
        super(DCENet, self).__init__()
        self.use_attention = use_attention

        self.conv1 = nn.Conv2d(3, 32, 3, 1, 1)
        self.conv2 = nn.Conv2d(32, 32, 3, 1, 1)
        self.conv3 = nn.Conv2d(32, 32, 3, 1, 1)

        if use_attention:
            self.se = SEBlock(32, reduction=8)

        self.conv4 = nn.Conv2d(32, 32, 3, 1, 1)
        self.conv5 = nn.Conv2d(64, 32, 3, 1, 1)
        self.conv6 = nn.Conv2d(64, 32, 3, 1, 1)
        self.conv7 = nn.Conv2d(64, 24, 3, 1, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x1 = self.relu(self.conv1(x))
        x2 = self.relu(self.conv2(x1))
        x3 = self.relu(self.conv3(x2))

        if self.use_attention:
            x3 = self.se(x3)

        x4 = self.relu(self.conv4(x3))
        x5 = torch.cat([x4, x3], 1)
        x5 = self.relu(self.conv5(x5))
        x6 = torch.cat([x5, x2], 1)
        x6 = self.relu(self.conv6(x6))
        x7 = torch.cat([x6, x1], 1)
        alpha = self.conv7(x7)
        return alpha

print('SE-Block and DCENet defined')

SE-Block and DCENet defined


In [21]:
# Evaluation - NO NIQE (PIQ library issue)
test_metrics = {'psnr': [], 'ssim': [], 'lpips': []}
enhanced_images = []
low_images = []
normal_images = []

# Try to load LPIPS
try:
    import lpips
    LPIPS_AVAILABLE = True
    lpips_model = lpips.LPIPS(net='alex', verbose=False).to(device)
    lpips_model.eval()
    print('LPIPS loaded')
except:
    LPIPS_AVAILABLE = False
    print('LPIPS not available (optional)')

print("\nEvaluating on LOL-v2 test set...")

with torch.no_grad():
    pbar = tqdm(test_loader, desc="Testing")
    for batch in pbar:
        low_imgs = batch['low'].to(device)
        normal_imgs = batch['normal'].to(device)

        alpha = model(low_imgs)
        enhanced = enhance_image_with_curves(low_imgs, alpha)

        enhanced_np = enhanced.cpu().permute(0, 2, 3, 1).numpy()
        normal_np = normal_imgs.cpu().permute(0, 2, 3, 1).numpy()
        low_np = low_imgs.cpu().permute(0, 2, 3, 1).numpy()

        for i in range(len(batch['low'])):
            psnr = psnr_metric(np.uint8(np.clip(normal_np[i] * 255, 0, 255)), 
                             np.uint8(np.clip(enhanced_np[i] * 255, 0, 255)), data_range=255)
            ssim = ssim_metric(np.uint8(np.clip(normal_np[i] * 255, 0, 255)),
                             np.uint8(np.clip(enhanced_np[i] * 255, 0, 255)), channel_axis=2, data_range=255)

            test_metrics['psnr'].append(psnr)
            test_metrics['ssim'].append(ssim)

            if LPIPS_AVAILABLE:
                img1_tensor = torch.from_numpy(enhanced_np[i]).permute(2, 0, 1).unsqueeze(0).float().to(device) * 2 - 1
                img2_tensor = torch.from_numpy(normal_np[i]).permute(2, 0, 1).unsqueeze(0).float().to(device) * 2 - 1
                lpips_val = lpips_model(img1_tensor, img2_tensor).item()
                test_metrics['lpips'].append(lpips_val)

            enhanced_images.append(enhanced_np[i])
            low_images.append(low_np[i])
            normal_images.append(normal_np[i])

        pbar.set_postfix({
            'PSNR': f"{np.mean(test_metrics['psnr'][-len(batch['low']):]):.2f}",
            'SSIM': f"{np.mean(test_metrics['ssim'][-len(batch['low']):]):.3f}"
        })

print("\nEvaluation completed!")

LPIPS loaded

Evaluating on LOL-v2 test set...


Testing: 100%|██████████| 13/13 [00:15<00:00,  1.22s/it, PSNR=8.90, SSIM=0.235]


Evaluation completed!


In [22]:
BASE_PATH = Path('LOL-v2')
test_pairs = load_image_pairs(BASE_PATH / 'Real_captured' / 'Test')
test_dataset = LOLDataset(test_pairs, img_size=512, augment=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)

print(f"✓ Loaded test set: {len(test_pairs)} images")

✓ Loaded test set: 100 images


In [ ]:
# Curve enhancement and dataset functions
def enhance_image_with_curves(x, alpha):
    enhanced = x.clone()
    for i in range(8):
        alpha_i = alpha[:, i*3:(i+1)*3, :, :]
        enhanced = enhanced + alpha_i * enhanced * (1 - enhanced)
    return torch.clamp(enhanced, 0, 1)

class LOLDataset(Dataset):
    def __init__(self, image_pairs, img_size=512):
        self.image_pairs = image_pairs
        self.img_size = img_size

    def __len__(self):
        return len(self.image_pairs)

    def __getitem__(self, idx):
        pair = self.image_pairs[idx]
        low_img = Image.open(pair['low']).convert('RGB')
        normal_img = Image.open(pair['normal']).convert('RGB')

        resize_size = int(self.img_size * 1.2)
        low_img = transforms.functional.resize(low_img, (resize_size, resize_size), interpolation=Image.BILINEAR)
        normal_img = transforms.functional.resize(normal_img, (resize_size, resize_size), interpolation=Image.BILINEAR)

        low_img = transforms.functional.center_crop(low_img, (self.img_size, self.img_size))
        normal_img = transforms.functional.center_crop(normal_img, (self.img_size, self.img_size))

        low_tensor = transforms.ToTensor()(low_img)
        normal_tensor = transforms.ToTensor()(normal_img)

        return {'low': low_tensor, 'normal': normal_tensor, 'filename': pair['filename']}

def load_image_pairs(data_path):
    low_path = data_path / 'Low'
    normal_path = data_path / 'Normal'
    low_images = sorted([f for f in os.listdir(low_path) if f.endswith(('.jpg', '.png'))])
    pairs = []
    for low_img in low_images:
        normal_img = low_img.replace('low', 'normal')
        pairs.append({
            'low': str(low_path / low_img),
            'normal': str(normal_path / normal_img),
            'filename': low_img
        })
    return pairs

# Load model
model = DCENet(use_attention=True).to(device)
model.load_state_dict(torch.load('best_zerodce_seblock_v2.pth', map_location=device)) # change filename for new version
model.eval()
print('Model loaded successfully')

# Load data
BASE_PATH = Path('LOL-v2')
test_pairs = load_image_pairs(BASE_PATH / 'Real_captured' / 'Test')
test_dataset = LOLDataset(test_pairs, img_size=512)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)
print(f'Test set loaded: {len(test_pairs)} images\n')

Model loaded successfully
Test set loaded: 100 images



In [ ]:
# Plot Metrics Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(test_metrics['psnr'], bins=15, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(19, color='green', linestyle='--', linewidth=2, label='Target: 19.00')
axes[0].axvline(np.mean(test_metrics['psnr']), color='red', linestyle='--', linewidth=2, label=f"Mean: {np.mean(test_metrics['psnr']):.2f}")
axes[0].set_xlabel('PSNR (dB)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('PSNR Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(test_metrics['ssim'], bins=15, edgecolor='black', alpha=0.7, color='coral')
axes[1].axvline(0.65, color='green', linestyle='--', linewidth=2, label='Target: 0.65')
axes[1].axvline(np.mean(test_metrics['ssim']), color='red', linestyle='--', linewidth=2, label=f"Mean: {np.mean(test_metrics['ssim']):.4f}")
axes[1].set_xlabel('SSIM')
axes[1].set_ylabel('Frequency')
axes[1].set_title('SSIM Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('metrics_distribution_v2.png', dpi=150) # change filename for new version
plt.show()
print("Saved: metrics_distribution_v2.png")

In [ ]:
# Visualize Enhancement Results
num_show = min(8, len(enhanced_images))
fig, axes = plt.subplots(num_show, 3, figsize=(15, 4*num_show))

for idx in range(num_show):
    axes[idx, 0].imshow(np.clip(low_images[idx], 0, 1))
    axes[idx, 0].set_title("Low-Light")
    axes[idx, 0].axis('off')

    axes[idx, 1].imshow(np.clip(enhanced_images[idx], 0, 1))
    psnr_val = test_metrics['psnr'][idx]
    ssim_val = test_metrics['ssim'][idx]
    axes[idx, 1].set_title(f"Enhanced\nPSNR: {psnr_val:.2f} | SSIM: {ssim_val:.3f}")
    axes[idx, 1].axis('off')

    axes[idx, 2].imshow(np.clip(normal_images[idx], 0, 1))
    axes[idx, 2].set_title("Ground Truth")
    axes[idx, 2].axis('off')

plt.suptitle('Low-Light Enhancement Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_results_v2.png', dpi=150) # change filename for new version
plt.show()
print("\nSaved: evaluation_results.png")

In [24]:
# Display Results
print("\n" + "="*70)
print("FINAL EVALUATION RESULTS")
print("="*70)

print(f"\nPSNR (Peak Signal-to-Noise Ratio):")
print(f"  Mean: {np.mean(test_metrics['psnr']):.2f} dB")
print(f"  Std:  {np.std(test_metrics['psnr']):.2f} dB")
print(f"  Target: > 19.00 dB")
print(f"  Status: {'PASS' if np.mean(test_metrics['psnr']) > 19 else 'FAIL'}")

print(f"\nSSIM (Structural Similarity):")
print(f"  Mean: {np.mean(test_metrics['ssim']):.4f}")
print(f"  Std:  {np.std(test_metrics['ssim']):.4f}")
print(f"  Target: > 0.65")
print(f"  Status: {'PASS' if np.mean(test_metrics['ssim']) > 0.65 else 'FAIL'}")

if LPIPS_AVAILABLE and len(test_metrics['lpips']) > 0:
    print(f"\nLPIPS (Perceptual Distance):")
    print(f"  Mean: {np.mean(test_metrics['lpips']):.4f}")
    print(f"  Std:  {np.std(test_metrics['lpips']):.4f}")
    print(f"  Target: < 0.30")
    print(f"  Status: {'PASS' if np.mean(test_metrics['lpips']) < 0.30 else 'FAIL'}")

print("="*70)


FINAL EVALUATION RESULTS

PSNR (Peak Signal-to-Noise Ratio):
  Mean: 9.64 dB
  Std:  1.64 dB
  Target: > 19.00 dB
  Status: FAIL

SSIM (Structural Similarity):
  Mean: 0.3875
  Std:  0.1287
  Target: > 0.65
  Status: FAIL

LPIPS (Perceptual Distance):
  Mean: 0.7163
  Std:  0.1630
  Target: < 0.30
  Status: FAIL
